# Step 0 · Hand check of the LLM labels

How reliable is the LLM used in `2-DataValidation.ipynb`? We take a random sample of families the LLM labelled **with a clear answer** (`pertinente` or `non_pertinente`), label them **by hand without looking at the LLM's answer**, and measure the agreement (percentage and Cohen's kappa).

- **No LLM calls, no cost:** the LLM answers are read from `validation/llm_cache.jsonl`; titles and abstracts come from PATSTAT.
- **Compares prompt versions:** every prompt version in the cache (v1-it: original Italian prompt; v2-en: English prompt + rule for generic aircraft technology) is compared with the same hand labels.
- **Blind:** the file to fill (`validation/hand_check.csv`) does not contain the LLM label.
- **Rule for the hand labels** (same as for the 'incerto' review): *pertinente* only if the text states or clearly implies an invention for unmanned aircraft; generic technologies where drones are only one possible use, manned aircraft, ground-based anti-drone systems and non-flying vehicles are *non_pertinente*.

Reading the result: Cohen's kappa above ~0.6 is usually read as substantial agreement, above ~0.8 as almost perfect (Landis & Koch, 1977).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import and_, func

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR

ENV = "TEST"                  # same environment as 1-DataPreparation
N_PER_LABEL = 10              # 10 'pertinente' + 10 'non_pertinente' from the LLM = 20 families
RANDOM_STATE = 42

VALIDATION_DIR = Path("validation")
LLM_CACHE = VALIDATION_DIR / "llm_cache.jsonl"
HAND_CHECK = VALIDATION_DIR / "hand_check.csv"

patstat = PatstatClient(env=ENV)
db = patstat.orm()

answers = pd.DataFrame([json.loads(line) for line in LLM_CACHE.read_text(encoding="utf-8").splitlines() if line.strip()])
if "prompt_name" not in answers.columns:
    answers["prompt_name"] = np.nan
answers["prompt_name"] = answers["prompt_name"].fillna("v1-it")   # answers saved before prompt names existed
print("LLM answers in the cache:", len(answers))
print(pd.crosstab(answers["prompt_name"], answers["llm_label"]).to_string())

## 1. Pick the families and write the file to fill

Run once: it writes `validation/hand_check.csv`. If the file already exists it is kept, so your labels are never overwritten.

In [ ]:
def family_texts(family_ids):
    "English title and the longest English abstract of each family (same rule as validate_sample)."
    q = (
        db.query(
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.appln_id,
            TLS202_APPLN_TITLE.appln_title,
            TLS203_APPLN_ABSTR.appln_abstract,
        )
        .select_from(TLS201_APPLN)
        .outerjoin(TLS202_APPLN_TITLE, and_(
            TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id,
            func.lower(func.trim(TLS202_APPLN_TITLE.appln_title_lg)) == "en",
        ))
        .outerjoin(TLS203_APPLN_ABSTR, and_(
            TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id,
            func.lower(func.trim(TLS203_APPLN_ABSTR.appln_abstract_lg)) == "en",
        ))
        .filter(TLS201_APPLN.docdb_family_id.in_([int(f) for f in family_ids]))
    )
    df = patstat.df(q)
    for column in ["appln_title", "appln_abstract"]:
        df[column] = df[column].fillna("").astype(str).str.strip()
    df["_len"] = df["appln_abstract"].str.len()
    return (
        df.sort_values(["_len", "appln_id"], ascending=[False, True])
        .drop_duplicates("docdb_family_id")[["docdb_family_id", "appln_title", "appln_abstract"]]
    )


if HAND_CHECK.exists():
    print("Already created, keeping your labels:", HAND_CHECK)
else:
    first = answers[answers["prompt_name"] == "v1-it"]
    decided = first[first["llm_label"].isin(["pertinente", "non_pertinente"])]
    pick = (
        decided.groupby("llm_label", group_keys=False)
        .apply(lambda g: g.sample(n=min(N_PER_LABEL, len(g)), random_state=RANDOM_STATE))
        .sample(frac=1, random_state=RANDOM_STATE)          # shuffle: the order does not reveal the LLM label
    )
    blind = (
        pick[["docdb_family_id"]]
        .merge(family_texts(pick["docdb_family_id"]), on="docdb_family_id", how="left")
        .assign(human_label="")
    )
    blind.to_csv(HAND_CHECK, index=False)
    print("Written:", HAND_CHECK, "| families:", len(blind))

hand = pd.read_csv(HAND_CHECK, keep_default_na=False)
for _, r in hand.iterrows():
    print(f"\n[{r['docdb_family_id']}] {r['appln_title']}\n  {r['appln_abstract'][:700]}")

## 2. Your labels

Read the families above **before** looking at anything else, and write one label per family: `"pertinente"` or `"non_pertinente"`. Then run the cell: the labels are saved into `validation/hand_check.csv`.

In [ ]:
LABELS = {
    72044684: "non_pertinente",
    33535776: "non_pertinente",
    64918518: "non_pertinente",
    38321522: "non_pertinente",
    73854713: "non_pertinente",
    73650023: "non_pertinente",
    49879141: "pertinente",
    50207190: "non_pertinente",
    69597174: "non_pertinente",
    73863785: "pertinente",
    74975033: "pertinente",
    70281298: "non_pertinente",
    76305693: "non_pertinente",
    71201509: "pertinente",
    40451317: "non_pertinente",
    61030624: "pertinente",
    67190470: "non_pertinente",
    37401490: "non_pertinente",
    63854688: "non_pertinente",
    69643247: "non_pertinente",
}

hand = pd.read_csv(HAND_CHECK, keep_default_na=False)
bad = {v for v in LABELS.values()} - {"pertinente", "non_pertinente"}
assert not bad, f"Use only 'pertinente' or 'non_pertinente', not {bad}"
hand["human_label"] = hand["docdb_family_id"].map(LABELS).fillna(hand["human_label"])
hand.to_csv(HAND_CHECK, index=False)
print("Still empty:", int((hand["human_label"].str.strip() == "").sum()), "of", len(hand))

## 3. Agreement between you and the LLM

In [ ]:
def cohen_kappa(a, b):
    "Cohen's kappa for two lists of labels."
    a, b = pd.Series(list(a)), pd.Series(list(b))
    po = (a == b).mean()
    pe = sum((a == c).mean() * (b == c).mean() for c in set(a) | set(b))
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


def wilson_interval(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return (centre - half, centre + half)


hand = pd.read_csv(HAND_CHECK, keep_default_na=False)
hand = hand[hand["human_label"].str.strip() != ""]
if hand.empty:
    print("Label the families first.")
else:
    rows, details = [], {}
    for version, group in answers.groupby("prompt_name"):
        done = hand.merge(group[["docdb_family_id", "llm_label", "reason"]], on="docdb_family_id")
        if done.empty:
            continue
        same = done["human_label"] == done["llm_label"]
        low, high = wilson_interval(int(same.sum()), len(done))
        rows.append({
            "prompt": version,
            "families checked": len(done),
            "agreement": same.mean(),
            "agreement 95% CI min": low,
            "agreement 95% CI max": high,
            "Cohen's kappa": cohen_kappa(done["human_label"], done["llm_label"]),
            "LLM said incerto": int(done["llm_label"].eq("incerto").sum()),
        })
        details[version] = done.assign(same=same)

    summary = pd.DataFrame(rows).set_index("prompt")
    summary.reset_index().to_csv(VALIDATION_DIR / "hand_check_summary.csv", index=False)
    display(summary.round(3))

    for version, done in details.items():
        print(f"\n=== {version}: human (rows) vs LLM (columns)")
        display(pd.crosstab(done["human_label"], done["llm_label"], rownames=["human"], colnames=["LLM"]))
        print("Disagreements (read them: who is right?):")
        display(done.loc[~done["same"], ["docdb_family_id", "appln_title", "human_label", "llm_label", "reason"]])